# Besoin Client 2 – Clustering selon la position

Objectif : partitionner les bornes selon leur position géographique (lat/lon) via un algorithme de clustering non-supervisé, trouver le nombre optimal de clusters et évaluer la qualité.

Responsable : **Personne 2**

## 1. Imports et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import joblib
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('IRVE_cleaned.csv', low_memory=False)
print(df.shape)
df.head()

## 2. Préparation des données

**Justification des colonnes choisies :** on utilise uniquement latitude et longitude car l'objectif est un clustering géographique.

In [ ]:
cols = ['consolidated_latitude', 'consolidated_longitude']
df_clust = df[cols].dropna()

# Normalisation
scaler = StandardScaler()
X = scaler.fit_transform(df_clust)
print(f'Shape : {X.shape}')

## 3. Recherche du nombre optimal de clusters

Méthode Elbow (inertie) pour KMeans.

In [ ]:
K_range = range(2, 12)
inertias = []
silhouettes = []

sample = X[:10000] if len(X) > 10000 else X  # sous-échantillon pour la vitesse

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(sample)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(sample, labels))
    print(f'k={k} | Inertia={km.inertia_:.0f} | Silhouette={silhouettes[-1]:.4f}')

# Elbow plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(K_range, inertias, 'o-'); ax1.set_title('Elbow – Inertie'); ax1.set_xlabel('k')
ax2.plot(K_range, silhouettes, 'o-', color='orange'); ax2.set_title('Silhouette score'); ax2.set_xlabel('k')
plt.tight_layout(); plt.show()

## 4. Entraînement du modèle final

**Justification du nombre de clusters choisi :** (à remplir après analyse du coude)

In [ ]:
K_OPTIMAL = 6  # ← changer après analyse

km_final = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df_clust['cluster'] = km_final.fit_predict(X)

# Sauvegarde du modèle
joblib.dump(km_final, 'model_clustering.pkl')
joblib.dump(scaler,   'scaler_clustering.pkl')
print('Modèles sauvegardés.')

## 5. Métriques d'évaluation

In [ ]:
labels = df_clust['cluster'].values
sil = silhouette_score(X, labels)
ch  = calinski_harabasz_score(X, labels)
db  = davies_bouldin_score(X, labels)

print(f'Silhouette Coefficient : {sil:.4f}  (plus proche de 1 = mieux)')
print(f'Calinski-Harabasz Index : {ch:.2f}  (plus élevé = mieux)')
print(f'Davies-Bouldin Index   : {db:.4f}  (plus proche de 0 = mieux)')

## 6. Visualisation sur carte

In [ ]:
palette = ['red','blue','green','purple','orange','cadetblue','darkred','pink']
carte = folium.Map(
    location=[df_clust['consolidated_latitude'].mean(),
               df_clust['consolidated_longitude'].mean()],
    zoom_start=6
)

sample_df = df_clust.sample(min(3000, len(df_clust)), random_state=42)
for _, row in sample_df.iterrows():
    folium.CircleMarker(
        location=[row['consolidated_latitude'], row['consolidated_longitude']],
        radius=3,
        color=palette[int(row['cluster']) % len(palette)],
        fill=True,
        popup=f"Cluster {int(row['cluster'])}"
    ).add_to(carte)

carte.save('carte_clusters.html')
print('Carte sauvegardée : carte_clusters.html')
carte

## 7. Conclusion

- Algorithme choisi : **justification à rédiger**
- Nb de clusters retenu : **justification à rédiger**
- Interprétation des métriques : **à rédiger**